# Extract Rich IFC Metadata for Annotated Subset

**Workflow:**
1. Load GlobalIds from `data/annotated_subset.json` (our curated subset)
2. Look up IFC file paths from `data/metadata.json` (maps GlobalId → source file)
3. Open each IFC file and extract comprehensive metadata
4. Save enriched data to `data/annotated_metadata.json`

**Extracted fields beyond GlobalId and IfcType:**

- **Direct attributes**: Name, Description, ObjectType, Tag, PredefinedType
- **Type definition**: TypeName, TypeId, TypeDescription
- **Spatial container**: ContainerName, ContainerType, Elevation
- **Material**: MaterialName
- **Property sets**: IsExternal, LoadBearing, FireRating, Reference, Status
- **IFC Quantities**: IFC_Length, IFC_Width, IFC_Height, IFC_Area, IFC_Volume
- **Relational flags**: HasOpenings, FillsVoid, PartOfAssembly, HasCoverings, NumConnections

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
import pandas as pd

import ifcopenshell
import ifcopenshell.util.element as util_element
import ifcopenshell.util.placement as util_placement

In [ ]:
# Paths
PROJECT_ROOT = Path('../../')
ANNOTATED_SUBSET = PROJECT_ROOT / 'data' / 'annotated_subset.json'
METADATA_JSON = PROJECT_ROOT / 'data' / 'metadata.json'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'processed_data' / 'features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Annotated subset: {ANNOTATED_SUBSET}")
print(f"Metadata: {METADATA_JSON}")

## 1. Load Annotated Subset and Build GlobalId → IFC Path Mapping

In [ ]:
# Load annotated subset
with open(ANNOTATED_SUBSET) as f:
    annotated = json.load(f)

annotated_gids = {d['GlobalId'] for d in annotated}
print(f"Annotated objects: {len(annotated_gids)}")

# Load full metadata to get source IFC paths
with open(METADATA_JSON) as f:
    metadata = json.load(f)

print(f"Total objects in metadata: {len(metadata)}")

In [ ]:
# Build mapping: GlobalId -> (relative IFC path, mesh_filename)
gid_to_source = {}
for item in metadata:
    gid = item['GlobalId']
    if gid in annotated_gids:
        gid_to_source[gid] = {
            'relative_source_path': item['relative_source_path'],
            'mesh_filename': item['mesh_filename']
        }

print(f"Matched {len(gid_to_source)} / {len(annotated_gids)} annotated objects to IFC files")

# Group by IFC file to minimize file opens
ifc_to_gids = defaultdict(list)
for gid, info in gid_to_source.items():
    ifc_to_gids[info['relative_source_path']].append(gid)

print(f"Unique IFC files to process: {len(ifc_to_gids)}")

## 2. Define Rich Metadata Extraction Function

In [ ]:
from collections import defaultdict

def extract_rich_metadata(element, model):
    """
    Extract comprehensive metadata from an IFC element.
    
    Returns a dict with fields in same order as metadata.json, plus additional attributes.
    """
    meta = {}
    
    # Basic identifiers (same as metadata.json)
    meta['GlobalId'] = element.GlobalId
    meta['IfcType'] = element.is_a()
    
    # === Direct attributes ===
    meta['Name'] = element.Name
    meta['Description'] = element.Description
    meta['ObjectType'] = element.ObjectType
    meta['Tag'] = getattr(element, 'Tag', None)
    
    # PredefinedType
    try:
        meta['PredefinedType'] = util_element.get_predefined_type(element)
    except:
        meta['PredefinedType'] = None
    
    # === Type definition ===
    try:
        elem_type = util_element.get_type(element)
        if elem_type:
            meta['TypeName'] = elem_type.Name
            meta['TypeId'] = elem_type.GlobalId
            meta['TypeDescription'] = elem_type.Description
        else:
            meta['TypeName'] = None
            meta['TypeId'] = None
            meta['TypeDescription'] = None
    except:
        meta['TypeName'] = None
        meta['TypeId'] = None
        meta['TypeDescription'] = None
    
    # === Spatial container ===
    try:
        container = util_element.get_container(element)
        if container:
            meta['ContainerName'] = container.Name
            meta['ContainerType'] = container.is_a()
            meta['Elevation'] = getattr(container, 'Elevation', None)
        else:
            meta['ContainerName'] = None
            meta['ContainerType'] = None
            meta['Elevation'] = None
    except:
        meta['ContainerName'] = None
        meta['ContainerType'] = None
        meta['Elevation'] = None
    
    # === Material ===
    try:
        material = util_element.get_material(element)
        if material:
            if hasattr(material, 'Name'):
                meta['MaterialName'] = material.Name
            elif hasattr(material, 'ForLayerSet'):
                layers = material.ForLayerSet.MaterialLayers
                meta['MaterialName'] = layers[0].Material.Name if layers else None
            elif hasattr(material, 'MaterialLayers'):
                layers = material.MaterialLayers
                meta['MaterialName'] = layers[0].Material.Name if layers else None
            else:
                meta['MaterialName'] = str(type(material).__name__)
        else:
            meta['MaterialName'] = None
    except:
        meta['MaterialName'] = None
    
    # === Property sets ===
    try:
        psets = util_element.get_psets(element, psets_only=True)
        for pset_name, props in psets.items():
            if 'IsExternal' in props:
                meta['IsExternal'] = props['IsExternal']
            if 'LoadBearing' in props:
                meta['LoadBearing'] = props['LoadBearing']
            if 'FireRating' in props:
                meta['FireRating'] = props['FireRating']
            if 'Reference' in props:
                meta['Reference'] = props['Reference']
            if 'Status' in props:
                meta['Status'] = props['Status']
            if 'AcousticRating' in props:
                meta['AcousticRating'] = props['AcousticRating']
            if 'ThermalTransmittance' in props:
                meta['ThermalTransmittance'] = props['ThermalTransmittance']
    except:
        pass
    
    for key in ['IsExternal', 'LoadBearing', 'FireRating', 'Reference', 'Status', 
                'AcousticRating', 'ThermalTransmittance']:
        if key not in meta:
            meta[key] = None
    
    # === IFC Quantities ===
    try:
        qtos = util_element.get_psets(element, qtos_only=True)
        for qto_name, quantities in qtos.items():
            for key in ['Length', 'Width', 'Height', 'Area', 'Volume', 
                        'GrossArea', 'NetArea', 'GrossVolume', 'NetVolume',
                        'GrossLength', 'NetLength', 'Perimeter']:
                if key in quantities and quantities[key] is not None:
                    meta[f'IFC_{key}'] = quantities[key]
    except:
        pass
    
    # === Basic Relational flags ===
    try:
        meta['HasOpenings'] = len(element.HasOpenings) > 0 if hasattr(element, 'HasOpenings') else False
        meta['FillsVoid'] = len(element.FillsVoids) > 0 if hasattr(element, 'FillsVoids') else False
        
        try:
            aggregate = util_element.get_aggregate(element)
            meta['PartOfAssembly'] = aggregate is not None
            meta['AssemblyType'] = aggregate.is_a() if aggregate else None
        except:
            meta['PartOfAssembly'] = False
            meta['AssemblyType'] = None
            
        meta['HasCoverings'] = len(element.HasCoverings) > 0 if hasattr(element, 'HasCoverings') else False
        meta['NumConnections'] = len(element.ConnectedTo) if hasattr(element, 'ConnectedTo') else 0
            
    except:
        meta['HasOpenings'] = None
        meta['FillsVoid'] = None
        meta['PartOfAssembly'] = None
        meta['AssemblyType'] = None
        meta['HasCoverings'] = None
        meta['NumConnections'] = None
    
    # === Connectivity Features ===
    try:
        # Count connections by IFC type
        connected_types = defaultdict(int)
        
        if hasattr(element, 'ConnectedTo'):
            for rel in element.ConnectedTo:
                try:
                    connected_types[rel.RelatedElement.is_a()] += 1
                except:
                    pass
                    
        if hasattr(element, 'ConnectedFrom'):
            for rel in element.ConnectedFrom:
                try:
                    connected_types[rel.RelatingElement.is_a()] += 1
                except:
                    pass
        
        # Connections to specific types
        meta['ConnectedToWall'] = connected_types.get('IfcWall', 0) + connected_types.get('IfcWallStandardCase', 0)
        meta['ConnectedToSlab'] = connected_types.get('IfcSlab', 0)
        meta['ConnectedToBeam'] = connected_types.get('IfcBeam', 0)
        meta['ConnectedToColumn'] = connected_types.get('IfcColumn', 0)
        meta['ConnectedToRoof'] = connected_types.get('IfcRoof', 0)
        meta['NumUniqueConnectedTypes'] = len(connected_types)
        meta['ConnectedTypes'] = list(connected_types.keys()) if connected_types else []
        
    except:
        meta['ConnectedToWall'] = 0
        meta['ConnectedToSlab'] = 0
        meta['ConnectedToBeam'] = 0
        meta['ConnectedToColumn'] = 0
        meta['ConnectedToRoof'] = 0
        meta['NumUniqueConnectedTypes'] = 0
        meta['ConnectedTypes'] = []
    
    # === Void/Opening Relationships ===
    try:
        # What type of element's void does this fill? (for doors/windows)
        meta['FillsVoidInType'] = None
        if hasattr(element, 'FillsVoids') and element.FillsVoids:
            for rel in element.FillsVoids:
                try:
                    opening = rel.RelatingOpeningElement
                    if hasattr(opening, 'VoidsElements') and opening.VoidsElements:
                        voided_element = opening.VoidsElements[0].RelatingBuildingElement
                        meta['FillsVoidInType'] = voided_element.is_a()
                        break
                except:
                    pass
        
        # What types fill this element's openings? (for walls)
        opening_fillers = []
        if hasattr(element, 'HasOpenings'):
            for rel in element.HasOpenings:
                try:
                    opening = rel.RelatedOpeningElement
                    if hasattr(opening, 'HasFillings') and opening.HasFillings:
                        for fill_rel in opening.HasFillings:
                            filler = fill_rel.RelatedBuildingElement
                            opening_fillers.append(filler.is_a())
                except:
                    pass
        
        meta['HasDoorOpening'] = 'IfcDoor' in opening_fillers
        meta['HasWindowOpening'] = 'IfcWindow' in opening_fillers
        meta['OpeningFillerTypes'] = list(set(opening_fillers)) if opening_fillers else []
        meta['NumOpenings'] = len(element.HasOpenings) if hasattr(element, 'HasOpenings') else 0
        
    except:
        meta['FillsVoidInType'] = None
        meta['HasDoorOpening'] = False
        meta['HasWindowOpening'] = False
        meta['OpeningFillerTypes'] = []
        meta['NumOpenings'] = 0
    
    # === Space Boundaries ===
    try:
        meta['BoundsSpaces'] = False
        meta['NumBoundedSpaces'] = 0
        if hasattr(element, 'ProvidesBoundaries') and element.ProvidesBoundaries:
            meta['BoundsSpaces'] = True
            meta['NumBoundedSpaces'] = len(element.ProvidesBoundaries)
    except:
        meta['BoundsSpaces'] = False
        meta['NumBoundedSpaces'] = 0
    
    return meta

## 3. Process All IFC Files and Extract Metadata

In [ ]:
# Extract rich metadata for all annotated objects
all_rich_metadata = []
errors = []

for ifc_rel_path, gids in tqdm(ifc_to_gids.items(), desc="Processing IFC files"):
    ifc_path = DATA_DIR / ifc_rel_path
    
    if not ifc_path.exists():
        errors.append(f"File not found: {ifc_path}")
        continue
    
    try:
        model = ifcopenshell.open(str(ifc_path))
        
        # Build GlobalId -> element lookup for this file
        gid_to_element = {}
        for element in model.by_type('IfcProduct'):
            if element.GlobalId in gids:
                gid_to_element[element.GlobalId] = element
        
        # Extract metadata for each target element
        for gid in gids:
            if gid in gid_to_element:
                element = gid_to_element[gid]
                try:
                    rich_meta = extract_rich_metadata(element, model)
                    
                    # Build final dict with same field order as metadata.json
                    meta = {
                        'relative_source_path': gid_to_source[gid]['relative_source_path'],
                        'GlobalId': rich_meta['GlobalId'],
                        'IfcType': rich_meta['IfcType'],
                        'mesh_filename': gid_to_source[gid]['mesh_filename'],
                    }
                    # Add all other rich attributes
                    for key, val in rich_meta.items():
                        if key not in meta:
                            meta[key] = val
                    
                    all_rich_metadata.append(meta)
                except Exception as e:
                    errors.append(f"Error extracting {gid}: {e}")
            else:
                errors.append(f"GlobalId {gid} not found in {ifc_rel_path}")
                
    except Exception as e:
        errors.append(f"Error opening {ifc_path}: {e}")

print(f"\nExtracted metadata for {len(all_rich_metadata)} objects")
print(f"Errors: {len(errors)}")
if errors[:5]:
    print("Sample errors:")
    for e in errors[:5]:
        print(f"  - {e}")

## 4. Convert to DataFrame and Analyze Coverage

In [ ]:
df = pd.DataFrame(all_rich_metadata)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

In [ ]:
# Analyze coverage (non-null rate) for each column
coverage = (df.notna().sum() / len(df) * 100).round(1)
coverage_df = pd.DataFrame({
    'Column': coverage.index,
    'Coverage (%)': coverage.values,
    'Non-null Count': df.notna().sum().values
})
coverage_df = coverage_df.sort_values('Coverage (%)', ascending=False)
print("\nColumn Coverage:")
print(coverage_df.to_string(index=False))

In [ ]:
# Summary statistics for key fields
print("\n=== IFC Types ===")
print(df['IfcType'].value_counts())

print("\n=== Container Types ===")
print(df['ContainerType'].value_counts())

print("\n=== Materials (top 20) ===")
print(df['MaterialName'].value_counts().head(20))

print("\n=== Type Names (top 20) ===")
print(df['TypeName'].value_counts().head(20))

In [ ]:
# Boolean flags summary
bool_cols = ['HasOpenings', 'FillsVoid', 'PartOfAssembly', 'HasCoverings', 'IsExternal', 'LoadBearing']
print("\n=== Relational Flags Summary ===")
for col in bool_cols:
    if col in df.columns:
        true_count = df[col].sum() if df[col].dtype == bool else (df[col] == True).sum()
        print(f"{col}: {true_count} / {df[col].notna().sum()} ({true_count / max(df[col].notna().sum(), 1) * 100:.1f}%)")

In [ ]:
# IFC Quantities coverage per type
qty_cols = [c for c in df.columns if c.startswith('IFC_')]
print(f"\n=== IFC Quantity Columns ({len(qty_cols)}) ===")
print(qty_cols)

if qty_cols:
    print("\nQuantity coverage by IFC type:")
    qty_coverage = df.groupby('IfcType')[qty_cols].apply(lambda x: x.notna().mean() * 100).round(1)
    print(qty_coverage)

## 5. Save Enriched Metadata

In [ ]:
# Save as JSON (same format as metadata.json - indent=4)
output_json = DATA_DIR / 'annotated_metadata.json'
with open(output_json, 'w') as f:
    json.dump(all_rich_metadata, f, indent=4, default=str)
print(f"Saved JSON: {output_json}")

# Also save as Parquet (for efficient loading in analysis)
output_parquet = OUTPUT_DIR / 'annotated_metadata.parquet'
df.to_parquet(output_parquet, index=False)
print(f"Saved Parquet: {output_parquet}")

In [ ]:
# Preview
print("\nSample records:")
display(df.head(10))

## 6. Feature Summary Table

Summary of extracted features and their potential use in clustering:

In [ ]:
feature_summary = """
| Category | Feature | Coverage | Clustering Potential |
|----------|---------|----------|---------------------|
| Identity | GlobalId | 100% | Unique ID |
| Identity | IfcType | 100% | Ground truth label |
| Semantic | Name | varies | Type hints in naming |
| Semantic | TypeName | varies | Specific product definition |
| Semantic | PredefinedType | varies | Fine-grained type |
| Spatial | ContainerName | varies | Location context |
| Spatial | Elevation | varies | Z-position |
| Material | MaterialName | varies | Material composition |
| Property | IsExternal | varies | Exterior vs interior |
| Property | LoadBearing | varies | Structural role |
| Quantity | IFC_Length/Width/Height | varies | IFC-native dimensions |
| Relation | HasOpenings | varies | Has voids (doors have, columns don't) |
| Relation | FillsVoid | varies | Is a filler (door/window) |
| Relation | PartOfAssembly | varies | Assembly membership |
"""
print(feature_summary)